<a href="https://colab.research.google.com/github/Kibet-Rotich/DeepLearning/blob/master/Pytorch_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Hardware-Agnostic Setup

PyTorch allows tensor operations to run either on the host CPU or on accelerated hardware (NVIDIA CUDA GPUs, Apple Silicon MPS).

When you move a tensor or model to a GPU via `.to(device)`, PyTorch copies its memory from host RAM to GPU VRAM, allowing massive parallel SIMD execution.

We configure dynamic hardware selection below so this notebook executes seamlessly on any runtime.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from PIL import Image

# Dynamic hardware selection
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Active compute device: {device}")

Active compute device: cuda


# Section 2: Data Loading Architecture

PyTorch decouples data access into two abstractions:

1. **`torch.utils.data.Dataset`**: Stores sample metadata and defines sample-level fetching via `__getitem__(idx)` and `__len__()`. Data is loaded lazily from disk on demand, preventing RAM overflow.
2. **`torch.utils.data.DataLoader`**: An iterable wrapper that orchestrates batching, shuffling, multi-process loading (`num_workers`), and page-locked memory staging (`pin_memory=True`) for fast GPU transfer.

Below, we load the standard **FashionMNIST** dataset using `torchvision.transforms.v2` for image preprocessing.

In [ ]:
# Image preprocessing: Convert PIL images to scaled float tensors [0.0, 1.0]
transform_pipeline = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

# Download datasets lazily to disk
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform_pipeline
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform_pipeline
)

# Instantiate streaming data loaders
BATCH_SIZE = 64
train_dataloader = DataLoader(training_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

# Inspect batch dimensions: [Batch, Channels, Height, Width]
for X, y in train_dataloader:
    print(f"Batch X shape [N, C, H, W]: {X.shape}")
    print(f"Batch y shape: {y.shape} | Data type: {y.dtype}")
    break

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 212kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.95MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.6MB/s]


Batch X shape [N, C, H, W]: torch.Size([64, 1, 28, 28])
Batch y shape: torch.Size([64]) | Data type: torch.int64


### Custom Dataset Implementation
To load custom domain data (e.g., sensor arrays, local images, tabular records), inherit from `Dataset` and implement `__len__` and `__getitem__`.

In [ ]:
class CustomSensorDataset(Dataset):
    """Example custom dataset for structured tabular or sensor data."""
    def __init__(self, num_samples: int = 1000, num_features: int = 6):
        self.num_samples = num_samples
        self.features = torch.randn(num_samples, num_features)
        self.labels = torch.randint(0, 4, (num_samples,))

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        return self.features[idx], self.labels[idx]

sensor_dataset = CustomSensorDataset(num_samples=20, num_features=4)
sensor_loader = DataLoader(sensor_dataset, batch_size=4, shuffle=True)

for batch_idx, (bx, by) in enumerate(sensor_loader):
    print(f"Sample Batch {batch_idx} -> X: {bx.shape}, y: {by.shape}")
    if batch_idx == 1:
        break

Sample Batch 0 -> X: torch.Size([4, 4]), y: torch.Size([4])
Sample Batch 1 -> X: torch.Size([4, 4]), y: torch.Size([4])


# Section 3: Defining Neural Networks with `nn.Module`

Every neural network in PyTorch subclasses `nn.Module`.

* **`super().__init__()`**: Mandatory constructor call. Registers sub-modules, parameters, and gradient tracking hooks.
* **`nn.Flatten()`**: Flattens 2D spatial dimensions `(28x28)` into a 1D vector `(784)` per sample, preserving the leading batch dimension.
* **`nn.Sequential`**: Cascades layers in sequential execution order.
* **`forward(x)`**: Dictates data flow during computation.

In [ ]:
class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)  # 10 output logits (one per FashionMNIST category)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Instantiate and transfer weights to target device
model = FashionClassifier().to(device)
print(model)

FashionClassifier(
  (net): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=512, bias=True)
    (2): ReLU()
    (3): Linear(in_features=512, out_features=512, bias=True)
    (4): ReLU()
    (5): Linear(in_features=512, out_features=10, bias=True)
  )
)


# Section 4: Optimization, Autograd, and Training Loops

Every batch iteration follows a 5-step sequence:
1. `optimizer.zero_grad()`: Resets parameter gradient buffers.
2. `pred = model(X)`: Forward pass building the dynamic autograd computation graph.
3. `loss = loss_fn(pred, y)`: Computes prediction error.
4. `loss.backward()`: Reverse-mode autograd traverses the graph via the Chain Rule, calculating $\nabla_{\theta} L$.
5. `optimizer.step()`: Updates parameter values using calculated gradients.

### Training vs. Evaluation Modes
* `model.train()`: Enables regularization layers like Dropout and updates BatchNorm running stats.
* `model.eval()` + `torch.no_grad()`: Freezes statistical tracking and disables autograd graph caching to accelerate inference and conserve VRAM.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)

def train_epoch(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # 1. Reset gradients
        optimizer.zero_grad()

        # 2. Forward pass
        pred = model(X)
        loss = loss_fn(pred, y)

        # 3. Backward pass & parameter update
        loss.backward()
        optimizer.step()

        if batch % 200 == 0:
            loss_val, current = loss.item(), (batch + 1) * len(X)
            print(f"Train Loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")

def evaluate(dataloader, model, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()

    test_loss, correct = 0.0, 0.0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy = (correct / size) * 100
    print(f"Validation Metrics -> Accuracy: {accuracy:>0.1f}%, Avg Loss: {test_loss:>8f}\n")

# Execute end-to-end training
EPOCHS = 5
for epoch in range(EPOCHS):
    print(f"--- Epoch {epoch + 1} of {EPOCHS} ---")
    train_epoch(train_dataloader, model, loss_fn, optimizer, device)
    evaluate(test_dataloader, model, loss_fn, device)

print("FashionMNIST Model Training Complete.")

--- Epoch 1 of 5 ---
Train Loss: 2.305536  [   64/60000]
Train Loss: 1.950602  [12864/60000]
Train Loss: 1.228739  [25664/60000]
Train Loss: 0.932604  [38464/60000]
Train Loss: 0.791209  [51264/60000]
Validation Metrics -> Accuracy: 70.2%, Avg Loss: 0.797851

--- Epoch 2 of 5 ---
Train Loss: 0.830379  [   64/60000]
Train Loss: 0.708236  [12864/60000]
Train Loss: 0.658029  [25664/60000]
Train Loss: 0.708546  [38464/60000]
Train Loss: 0.583388  [51264/60000]
Validation Metrics -> Accuracy: 77.9%, Avg Loss: 0.629773

--- Epoch 3 of 5 ---
Train Loss: 0.574597  [   64/60000]
Train Loss: 0.532676  [12864/60000]
Train Loss: 0.618814  [25664/60000]
Train Loss: 0.449160  [38464/60000]
Train Loss: 0.498675  [51264/60000]
Validation Metrics -> Accuracy: 80.3%, Avg Loss: 0.562629

--- Epoch 4 of 5 ---
Train Loss: 0.441901  [   64/60000]
Train Loss: 0.593282  [12864/60000]
Train Loss: 0.528251  [25664/60000]
Train Loss: 0.443311  [38464/60000]
Train Loss: 0.723820  [51264/60000]
Validation Metrics 

# Section 5: Hands-On Exercises

### Exercise 1: Continuous Regression (Housing Price Predictor)
Predict a continuous target value given 3 features (Square Footage, Bedrooms, Age) using an MLP trained on Mean Squared Error (`nn.MSELoss`).

In [ ]:
# 1. Dataset definition
class HousingDataset(Dataset):
    def __init__(self, n_samples=1000):
        self.X = torch.randn(n_samples, 3)
        noise = torch.randn(n_samples, 1) * 0.1
        self.y = (3.5 * self.X[:, 0:1] + 1.2 * self.X[:, 1:2] - 0.8 * self.X[:, 2:3] + noise)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

housing_loader = DataLoader(HousingDataset(1000), batch_size=32, shuffle=True)

# 2. Architecture
class HousingRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.net(x)

regressor = HousingRegressor().to(device)
criterion = nn.MSELoss()
reg_optimizer = torch.optim.Adam(regressor.parameters(), lr=0.01)

# 3. Training Loop
regressor.train()
for epoch in range(10):
    total_loss = 0.0
    for bx, by in housing_loader:
        bx, by = bx.to(device), by.to(device)
        reg_optimizer.zero_grad()
        loss = criterion(regressor(bx), by)
        loss.backward()
        reg_optimizer.step()
        total_loss += loss.item() * bx.size(0)

    if (epoch + 1) % 2 == 0:
        print(f"Housing Model Epoch [{epoch+1:02d}/10] - MSE Loss: {total_loss / 1000:.4f}")

Housing Model Epoch [02/10] - MSE Loss: 0.5114
Housing Model Epoch [04/10] - MSE Loss: 0.0392
Housing Model Epoch [06/10] - MSE Loss: 0.0259
Housing Model Epoch [08/10] - MSE Loss: 0.0226
Housing Model Epoch [10/10] - MSE Loss: 0.0211


### Exercise 2: Multi-Class Sensor Fault Classification
Classify 6 continuous sensor readings into 4 discrete failure modes using `nn.CrossEntropyLoss`.

In [ ]:
# 1. Custom multi-class dataset
class SensorFaultDataset(Dataset):
    def __init__(self, num_samples: int = 1000):
        self.num_samples = num_samples
        self.features = torch.randn(num_samples, 6)
        self.labels = torch.randint(0, 4, (num_samples,))

    def __len__(self) -> int:
        return self.num_samples

    def __getitem__(self, idx: int):
        return self.features[idx], self.labels[idx]

fault_loader = DataLoader(SensorFaultDataset(1000), batch_size=64, shuffle=True)

# 2. Classifier network
class FaultClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(6, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 4)  # Raw unnormalized logits for 4 classes
        )

    def forward(self, x):
        return self.layers(x)

fault_model = FaultClassifier().to(device)
fault_loss_fn = nn.CrossEntropyLoss()
fault_optimizer = torch.optim.Adam(fault_model.parameters(), lr=0.01)

# 3. Training Loop
fault_model.train()
for epoch in range(10):
    epoch_loss = 0.0
    for bx, by in fault_loader:
        bx, by = bx.to(device), by.to(device)
        fault_optimizer.zero_grad()
        loss = fault_loss_fn(fault_model(bx), by)
        loss.backward()
        fault_optimizer.step()
        epoch_loss += loss.item() * bx.size(0)

    if (epoch + 1) % 2 == 0:
        print(f"Fault Model Epoch [{epoch+1:02d}/10] - CrossEntropy Loss: {epoch_loss / 1000:.4f}")

Fault Model Epoch [02/10] - CrossEntropy Loss: 1.3817
Fault Model Epoch [04/10] - CrossEntropy Loss: 1.3686
Fault Model Epoch [06/10] - CrossEntropy Loss: 1.3549
Fault Model Epoch [08/10] - CrossEntropy Loss: 1.3449
Fault Model Epoch [10/10] - CrossEntropy Loss: 1.3288


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hardware selection
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)

class SyntheticDataset(Dataset):
    def __init__(self, n_samples=5000, n_features=12, n_classes=3):
        self.features = torch.randn(n_samples, n_features)
        self.labels = torch.randint(0, n_classes, (n_samples,))

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.Dropout(p=0.2),
            nn.Linear(32, 3)
        )

    def forward(self, x):
        return self.layers(x)


def train_epoch(dataloader, model, loss_fn, optimizer, device):
    model.train()
    total_loss, correct, total_samples = 0.0, 0, 0

    for x, y in dataloader:
        x, y = x.to(device), y.to(device)

        # Forward pass
        pred = model(x)
        loss = loss_fn(pred, y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Metrics tracking (.item() prevents memory leaks from computation graphs)
        total_loss += loss.item() * x.size(0)
        correct += (pred.argmax(dim=1) == y).sum().item()
        total_samples += x.size(0)

    return total_loss / total_samples, correct / total_samples

def evaluate(dataloader, model, loss_fn, device):
    model.eval()
    total_loss, correct, total_samples = 0.0, 0, 0

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            pred = model(x)
            loss = loss_fn(pred, y)

            total_loss += loss.item() * x.size(0)
            correct += (pred.argmax(dim=1) == y).sum().item()
            total_samples += x.size(0)

    return total_loss / total_samples, correct / total_samples

# Model setup & training execution
model = Classifier().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# DataLoaders
train_loader = DataLoader(SyntheticDataset(4000), batch_size=32, shuffle=True)
val_loader = DataLoader(SyntheticDataset(1000), batch_size=32, shuffle=False)


for epoch in range(1, 6):
    train_loss, train_acc = train_epoch(train_loader, model, loss_fn, optimizer, device)
    val_loss, val_acc = evaluate(val_loader, model, loss_fn, device)
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f}, Acc: {train_acc:.2%} | Val Loss: {val_loss:.4f}, Acc: {val_acc:.2%}")

Epoch 01 | Train Loss: 1.1032, Acc: 33.85% | Val Loss: 1.1042, Acc: 32.10%
Epoch 02 | Train Loss: 1.0980, Acc: 35.73% | Val Loss: 1.1050, Acc: 33.00%
Epoch 03 | Train Loss: 1.0965, Acc: 36.70% | Val Loss: 1.1069, Acc: 32.20%
Epoch 04 | Train Loss: 1.0957, Acc: 36.05% | Val Loss: 1.1079, Acc: 33.60%
Epoch 05 | Train Loss: 1.0928, Acc: 36.93% | Val Loss: 1.1085, Acc: 32.60%
